# CSE-CIC-IDS2018 — Data Size Scaling Experiment

**How much of our performance comes from the model, and how much just comes from having more data?**

Notebooks `04` and `06` tuned hyperparameters with Optuna and trained each winning model once, on the
full balanced training set. That tells us how good the models are, but not *why* — a model that scores
0.98 on 280,000 rows might score 0.97 on 25,000 rows, in which case three quarters of the data is
buying almost nothing.

This notebook answers that by re-running the **exact same models with the exact same hyperparameters**
across three training-set sizes:

| Subset | Rows | Roughly |
|---|---|---|
| **Small** | 25,000 | ~9% of the training data |
| **Medium** | 100,000 | ~36% |
| **Large** | all rows | 100% |

Three things are held constant so the comparison is fair:

1. **Hyperparameters are frozen.** They are read from the `*_best_params.json` files that notebooks
   04 and 06 wrote. Nothing is re-tuned here — re-tuning per subset would confound "more data" with
   "better-tuned", and the whole point is to isolate the effect of data volume.
2. **The test set never changes.** Every model at every size is scored on the same held-out
   `test_selected.parquet`. Only the *training* set shrinks.
3. **The random seed is fixed**, and subsets are drawn with **stratified** sampling, so the class
   proportions in the Small subset match those in the Large one.

> **A note on the MLP.** Notebook 06 oversampled rare classes up to 5,000 rows before training the
> MLP. That step is deliberately *not* repeated here: it would mean "25,000 rows" described a
> different amount of real data for the MLP than for the tree models, and the curves would no longer
> be comparable. The MLP still gets its required `StandardScaler`, refit on each subset (fitting it
> on the full data would leak information the small run is not supposed to have).

## 1. Setup

In [ ]:
import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

In [ ]:
# ---------------------------------------------------------------------------
# Project path.
#
# Set NIDS_PROJECT_PATH to point this notebook at your data folder, e.g.
#     Windows : set NIDS_PROJECT_PATH=M:\IDS\c_filesnew
#     macOS   : export NIDS_PROJECT_PATH=/Users/you/IDS/c_filesnew
#
# If it is unset we look for the data inside the repo, so the notebook runs on a
# fresh clone without editing any code.
# ---------------------------------------------------------------------------
def _locate_project_path():
    env = os.environ.get("NIDS_PROJECT_PATH")
    if env:
        return Path(env)

    here = Path.cwd()
    roots = [here, here.parent, here.parent.parent]
    candidates = []
    for root in roots:
        candidates += [root / "c_filesnew", root / "webapp_data"]
    candidates.append(Path(r"M:\IDS\c_filesnew"))

    for c in candidates:
        if (c / "Processed_Data" / "balanced_train_selected.parquet").exists():
            return c

    raise FileNotFoundError(
        "Could not find balanced_train_selected.parquet.\n"
        "Set NIDS_PROJECT_PATH to the folder containing Processed_Data/.\n"
        "If you just cloned the repo, the .parquet files are stored in Git LFS -- run:\n"
        "    git lfs install && git lfs pull"
    )


PROJECT_PATH = _locate_project_path()

PROCESSED_DATA_PATH = PROJECT_PATH / "Processed_Data"
RESULTS_PATH = PROJECT_PATH / "Results"
MODELS_PATH = RESULTS_PATH / "Models"
FIGURES_PATH = RESULTS_PATH / "Figures"
TABLES_PATH = RESULTS_PATH / "Tables"

for p in [MODELS_PATH, FIGURES_PATH, TABLES_PATH]:
    p.mkdir(parents=True, exist_ok=True)

print("Project path:", PROJECT_PATH)

In [ ]:
RANDOM_STATE = 42

# Training-set sizes to test. "Large" is filled in below once we know the row count.
SUBSET_SIZES = {
    "Small": 25_000,
    "Medium": 100_000,
    "Large": None,          # None == use every row
}

# The MLP is by far the slowest model here (notebook 06 took over two hours on the
# full set). Set to False to run the tree models only while iterating.
INCLUDE_MLP = True
MLP_MAX_ITER = 800          # matches FINAL_MAX_ITER in notebook 06

# Saved outputs
RESULTS_CSV = TABLES_PATH / "data_scaling_results.csv"
PER_CLASS_CSV = TABLES_PATH / "data_scaling_per_class_f1.csv"

## 2. Load the data

The training set shrinks; the test set never does. Loading both once, up front, means every run below is scored against an identical `X_test`.

In [ ]:
train = pd.read_parquet(PROCESSED_DATA_PATH / "balanced_train_selected.parquet")
test = pd.read_parquet(PROCESSED_DATA_PATH / "test_selected.parquet")

X_train_full = train.drop(columns=["Label"])
y_train_full = train["Label"]

X_test = test.drop(columns=["Label"])
y_test = test["Label"]

N_CLASSES = int(y_train_full.nunique())
SUBSET_SIZES["Large"] = len(X_train_full)

print("Full train :", X_train_full.shape)
print("Test       :", X_test.shape)
print("Classes    :", N_CLASSES)
print("\nSubset sizes to test:")
for name, n in SUBSET_SIZES.items():
    print(f"  {name:<8} {n:>8,} rows  ({n / len(X_train_full):6.1%})")

In [ ]:
# Readable class names for the rare-class analysis in section 8.
label_map = {}
mapping_file = PROCESSED_DATA_PATH / "label_mapping.csv"
if mapping_file.exists():
    _m = pd.read_csv(mapping_file)
    label_map = dict(zip(_m["Encoded"], _m["Class"]))
    print(f"Loaded {len(label_map)} class names.")
else:
    print("label_mapping.csv not found -- charts will show numeric labels.")

## 3. Frozen hyperparameters

Read straight from the JSON files notebooks 04 and 06 saved next to each trained model. If a file is missing, that model is skipped with a warning rather than silently trained on defaults — defaults would make the whole comparison meaningless.

In [ ]:
def load_best_params(model_name):
    path = MODELS_PATH / f"{model_name}_best_params.json"
    if not path.exists():
        print(f"  [skip] {model_name}: {path.name} not found")
        return None
    with open(path) as f:
        return json.load(f)


TREE_MODELS = {
    "RandomForest": "random_forest",
    "HistGradientBoosting": "hist_gradient_boosting",
    "XGBoost": "xgboost",
}

best_params = {}

for display_name in TREE_MODELS:
    params = load_best_params(f"{display_name}_Tuned")
    if params:
        best_params[display_name] = params

if INCLUDE_MLP:
    mlp_params = load_best_params("MLP_Tuned")
    if mlp_params:
        best_params["MLP"] = mlp_params

print()
for name, params in best_params.items():
    print(f"{name}:")
    for k, v in params.items():
        if k != "algorithm":
            print(f"    {k} = {v}")
    print()

## 4. Model builders

`instantiate_model` is the same helper used in notebook 04 — kept identical so a model built here is byte-for-byte the model built there. `build_mlp` reassembles the layer sizes from the flat `n_units_l0`, `n_units_l1`, … keys that Optuna stored.

In [ ]:
PREFIXES = {
    "decision_tree": "dt_",
    "random_forest": "rf_",
    "hist_gradient_boosting": "hgb_",
    "xgboost": "xgb_",
    "adaboost": "ada_",
}


def instantiate_model(algorithm, params, n_classes, random_state=RANDOM_STATE):
    """Build a ready-to-fit estimator from a prefixed parameter dict (as notebook 04 does)."""
    prefix = PREFIXES[algorithm]
    p = {k[len(prefix):]: v for k, v in params.items() if k.startswith(prefix)}

    if algorithm == "decision_tree":
        return DecisionTreeClassifier(**p, random_state=random_state)

    if algorithm == "random_forest":
        return RandomForestClassifier(**p, random_state=random_state, n_jobs=-1)

    if algorithm == "hist_gradient_boosting":
        return HistGradientBoostingClassifier(**p, random_state=random_state)

    if algorithm == "xgboost":
        return XGBClassifier(
            **p,
            objective="multi:softprob",
            num_class=n_classes,
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=random_state,
            n_jobs=-1,
        )

    if algorithm == "adaboost":
        base_depth = p.pop("base_max_depth")
        base = DecisionTreeClassifier(max_depth=base_depth, random_state=random_state)
        return AdaBoostClassifier(estimator=base, **p, random_state=random_state)

    raise ValueError(f"Unknown algorithm: {algorithm}")


def build_mlp(params, max_iter=MLP_MAX_ITER, random_state=RANDOM_STATE):
    """Rebuild notebook 06's MLP from its saved flat parameter dict."""
    n_layers = params["n_layers"]
    hidden_layer_sizes = tuple(params[f"n_units_l{i}"] for i in range(n_layers))

    return MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        activation=params["activation"],
        solver="adam",
        alpha=params["alpha"],
        learning_rate_init=params["learning_rate_init"],
        batch_size=params["batch_size"],
        max_iter=max_iter,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=15,
        random_state=random_state,
    )

## 5. Drawing the subsets

Stratified sampling keeps each class at the same proportion it has in the full set. That matters here because the classes are wildly unbalanced — `SQL Injection` has only 70 rows in the entire training set, so the Small subset will contain a handful at best. Section 8 comes back to what that does to the scores.

In [ ]:
def stratified_subsample(X, y, n_rows, random_state=RANDOM_STATE):
    """Take `n_rows` rows, preserving class proportions. Returns the full set if n_rows >= len(X)."""
    if n_rows >= len(X):
        return X, y

    X_sub, _, y_sub, _ = train_test_split(
        X, y,
        train_size=n_rows,
        stratify=y,
        random_state=random_state,
    )
    return X_sub, y_sub

In [ ]:
# How many rows of each class actually survive into each subset?
coverage = {}

for size_name, n_rows in SUBSET_SIZES.items():
    _, y_sub = stratified_subsample(X_train_full, y_train_full, n_rows)
    coverage[size_name] = y_sub.value_counts().sort_index()

coverage_df = pd.DataFrame(coverage).fillna(0).astype(int)
if label_map:
    coverage_df.index = [label_map.get(i, i) for i in coverage_df.index]
coverage_df.index.name = "Class"

print("Rows per class in each subset:\n")
print(coverage_df.sort_values("Large"))

missing = coverage_df[(coverage_df == 0).any(axis=1)]
if len(missing):
    print("\nWarning -- these classes are absent from at least one subset:")
    print(missing)

## 6. Run the experiment

For every (model, subset) pair: fit on the subset, predict the full test set, record the metrics. The MLP additionally gets a `StandardScaler` fit on **its own subset only** — fitting it on the full training data would hand the Small run statistics it has no business knowing.

**One wrinkle worth knowing about.** XGBoost's scikit-learn wrapper insists the training labels are a contiguous run `0, 1, … n-1`. As soon as a subset is small enough to lose a rare class entirely, the surviving labels have a hole in them (`SQL Injection` is encoded `13`, so we would hand it `[0…12, 14]`) and `fit()` raises `ValueError: Invalid classes inferred from unique values of y`. `run_one` therefore relabels to a contiguous range before fitting XGBoost and maps the predictions straight back afterwards, so the metrics stay in the original label space. The other models do not need this.

Note that a class missing from a subset is a genuine result, not an error to hide — the model cannot predict what it has never seen, so that class scores 0 and drags macro F1 down. That is precisely the effect this experiment is meant to expose.

In [ ]:
# Every class in the problem, so each subset produces the same rows in the per-class table
# even when a rare class is missing from that subset's training data.
ALL_CLASSES = np.sort(y_train_full.unique())


def run_one(model_name, params, size_name, n_rows):
    """Train one model on one subset and score it against the fixed test set."""
    X_sub, y_sub = stratified_subsample(X_train_full, y_train_full, n_rows)

    present = np.sort(y_sub.unique())
    missing = sorted(set(ALL_CLASSES) - set(present))
    if missing:
        names = [label_map.get(int(m), m) for m in missing] if label_map else missing
        print(f"  note: {len(missing)} class(es) absent from this subset -> {names}")

    y_fit = y_sub

    if model_name == "MLP":
        model = build_mlp(params)
        scaler = StandardScaler()
        X_fit = pd.DataFrame(scaler.fit_transform(X_sub), columns=X_sub.columns, index=X_sub.index)
        X_eval = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
    else:
        X_fit, X_eval = X_sub, X_test

        if TREE_MODELS[model_name] == "xgboost":
            # Relabel to a contiguous 0..k-1 range; undo it on the predictions below.
            to_contiguous = {orig: i for i, orig in enumerate(present)}
            from_contiguous = {i: orig for orig, i in to_contiguous.items()}
            y_fit = y_sub.map(to_contiguous)
            model = instantiate_model(TREE_MODELS[model_name], params, n_classes=len(present))
        else:
            from_contiguous = None
            model = instantiate_model(TREE_MODELS[model_name], params, n_classes=N_CLASSES)

    if model_name == "MLP":
        from_contiguous = None

    start = time.time()
    model.fit(X_fit, y_fit)
    train_time = time.time() - start

    start = time.time()
    y_pred = model.predict(X_eval)
    pred_time = time.time() - start

    if from_contiguous is not None:
        y_pred = pd.Series(y_pred).map(from_contiguous).to_numpy()

    row = {
        "Model": model_name,
        "Subset": size_name,
        "Train Rows": len(X_sub),
        "Classes Present": int(len(present)),
        "Classes Missing": int(len(missing)),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Macro Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "Training Time (s)": train_time,
        "Prediction Time (s)": pred_time,
    }

    # Per-class F1, kept in long form for the rare-class chart later. Passing `labels`
    # pins the row set so every (model, subset) pair reports the same classes.
    report = classification_report(
        y_test, y_pred, labels=ALL_CLASSES, output_dict=True, zero_division=0
    )
    per_class = [
        {
            "Model": model_name,
            "Subset": size_name,
            "Train Rows": len(X_sub),
            "Class": label_map.get(int(cls), cls) if label_map else cls,
            "F1": report[str(cls)]["f1-score"],
            "Support": report[str(cls)]["support"],
            "In Training Subset": cls in present,
        }
        for cls in ALL_CLASSES
    ]

    return row, per_class

The loop writes both CSVs after **every** run rather than at the end. A full sweep takes a long time — the MLP alone took over two hours on the full set in notebook 06 — and losing all of it to a failure on the last model would be painful. Cheap models are run first at each size so the headline numbers appear early.

In [ ]:
results_rows = []
per_class_rows = []

# Cheapest first, so useful results land early in a long run.
RUN_ORDER = ["HistGradientBoosting", "RandomForest", "XGBoost", "MLP"]
ordered_models = [m for m in RUN_ORDER if m in best_params]
ordered_models += [m for m in best_params if m not in RUN_ORDER]

total = len(ordered_models) * len(SUBSET_SIZES)
done = 0
sweep_start = time.time()

for size_name, n_rows in SUBSET_SIZES.items():
    for model_name in ordered_models:
        done += 1
        print("=" * 72)
        print(f"[{done}/{total}]  {model_name}  on  {size_name} ({n_rows:,} rows)")
        print("=" * 72)

        row, per_class = run_one(model_name, best_params[model_name], size_name, n_rows)

        results_rows.append(row)
        per_class_rows.extend(per_class)

        # Checkpoint after every run so a crash never costs the whole sweep.
        pd.DataFrame(results_rows).to_csv(RESULTS_CSV, index=False)
        pd.DataFrame(per_class_rows).to_csv(PER_CLASS_CSV, index=False)

        print(
            f"  Accuracy {row['Accuracy']:.4f}   "
            f"Macro F1 {row['Macro F1']:.4f}   "
            f"Macro Precision {row['Macro Precision']:.4f}   "
            f"Macro Recall {row['Macro Recall']:.4f}   "
            f"Weighted F1 {row['Weighted F1']:.4f}"
        )
        print(
            f"  trained in {row['Training Time (s)']:.1f}s, "
            f"predicted in {row['Prediction Time (s)']:.1f}s "
            f"| elapsed {(time.time() - sweep_start) / 60:.1f} min\n"
        )

print(f"All {total} runs complete in {(time.time() - sweep_start) / 60:.1f} minutes.")

## 7. Results

In [ ]:
results = pd.DataFrame(results_rows)

subset_order = list(SUBSET_SIZES.keys())
results["Subset"] = pd.Categorical(results["Subset"], categories=subset_order, ordered=True)
results = results.sort_values(["Model", "Subset"]).reset_index(drop=True)

results.to_csv(RESULTS_CSV, index=False)

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(PER_CLASS_CSV, index=False)

print(f"Saved -> {RESULTS_CSV.name}, {PER_CLASS_CSV.name}\n")
results

In [ ]:
# The headline table: macro F1 for every model at every size.
pivot_f1 = results.pivot(index="Model", columns="Subset", values="Macro F1")
pivot_acc = results.pivot(index="Model", columns="Subset", values="Accuracy")

print("Macro F1\n")
print(pivot_f1.round(4).to_string())
print("\n\nAccuracy\n")
print(pivot_acc.round(4).to_string())

In [ ]:
# How much did going from Small to Large actually buy?
gain = pd.DataFrame({
    "Macro F1 (Small)": pivot_f1["Small"],
    "Macro F1 (Large)": pivot_f1["Large"],
    "Gain": pivot_f1["Large"] - pivot_f1["Small"],
    "Accuracy (Small)": pivot_acc["Small"],
    "Accuracy (Large)": pivot_acc["Large"],
    "Accuracy Gain": pivot_acc["Large"] - pivot_acc["Small"],
}).sort_values("Gain", ascending=False)

gain.to_csv(TABLES_PATH / "data_scaling_gain_summary.csv")
gain.round(4)

## 8. Learning curves

If a line flattens out, extra data has stopped helping and the ceiling is the model, not the dataset. If it is still climbing at the right-hand edge, more data would likely still pay off.

In [ ]:
def plot_metric(metric, title, filename, logx=True):
    plt.figure(figsize=(9, 5.5))

    for model_name in sorted(results["Model"].unique()):
        sub = results[results["Model"] == model_name].sort_values("Train Rows")
        plt.plot(sub["Train Rows"], sub[metric], marker="o", linewidth=2, label=model_name)
        for _, r in sub.iterrows():
            plt.annotate(f"{r[metric]:.3f}", (r["Train Rows"], r[metric]),
                         textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)

    if logx:
        plt.xscale("log")
    plt.xlabel("Training rows (log scale)" if logx else "Training rows")
    plt.ylabel(metric)
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_PATH / filename, dpi=200)
    plt.show()


plot_metric("Macro F1", "Macro F1 vs Training Set Size", "data_scaling_macro_f1.png")

In [ ]:
plot_metric("Accuracy", "Accuracy vs Training Set Size", "data_scaling_accuracy.png")
plot_metric("Weighted F1", "Weighted F1 vs Training Set Size", "data_scaling_weighted_f1.png")

In [ ]:
# Cost side of the trade-off: what does the extra data cost in training time?
plt.figure(figsize=(9, 5.5))

for model_name in sorted(results["Model"].unique()):
    sub = results[results["Model"] == model_name].sort_values("Train Rows")
    plt.plot(sub["Train Rows"], sub["Training Time (s)"], marker="s", linewidth=2, label=model_name)

plt.xscale("log")
plt.yscale("log")
plt.xlabel("Training rows (log scale)")
plt.ylabel("Training time in seconds (log scale)")
plt.title("Training Cost vs Training Set Size")
plt.grid(alpha=0.3, which="both")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_PATH / "data_scaling_training_time.png", dpi=200)
plt.show()

## 9. What happens to the rare classes

Macro F1 weights every class equally, so it is dominated by the rare attacks. This is where shrinking the training set should hurt most — a class with 70 rows in total has almost nothing left once you take 9% of it.

In [ ]:
# Rare = smallest classes by training support.
class_totals = y_train_full.value_counts().sort_values()
rare_encoded = class_totals.head(5).index.tolist()
rare_names = [label_map.get(int(i), i) for i in rare_encoded] if label_map else rare_encoded

rare = per_class_df[per_class_df["Class"].isin(rare_names)]

rare_pivot = rare.pivot_table(index=["Class", "Model"], columns="Subset", values="F1")
rare_pivot = rare_pivot[[c for c in subset_order if c in rare_pivot.columns]]

print("Per-class F1 for the five rarest attack types:\n")
rare_pivot.round(4)

In [ ]:
fig, axes = plt.subplots(1, len(rare_names), figsize=(4.2 * len(rare_names), 4.5), sharey=True)
if len(rare_names) == 1:
    axes = [axes]

for ax, cls in zip(axes, rare_names):
    sub = rare[rare["Class"] == cls]
    for model_name in sorted(sub["Model"].unique()):
        s = sub[sub["Model"] == model_name].sort_values("Train Rows")
        ax.plot(s["Train Rows"], s["F1"], marker="o", label=model_name)
    ax.set_xscale("log")
    ax.set_title(str(cls), fontsize=10)
    ax.set_xlabel("Training rows")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("F1")
axes[-1].legend(fontsize=8)
plt.suptitle("Rare-Class F1 vs Training Set Size")
plt.tight_layout()
plt.savefig(FIGURES_PATH / "data_scaling_rare_class_f1.png", dpi=200)
plt.show()

## 10. Summary

In [ ]:
best_row = results.loc[results["Macro F1"].idxmax()]

print("=" * 72)
print("DATA SCALING EXPERIMENT -- SUMMARY")
print("=" * 72)
print(f"Models tested      : {', '.join(sorted(results['Model'].unique()))}")
print(f"Subset sizes       : {', '.join(f'{k} ({v:,})' for k, v in SUBSET_SIZES.items())}")
print(f"Test set           : {len(X_test):,} rows (identical for every run)")
print(f"Hyperparameters    : frozen, loaded from *_best_params.json")
print()
print(f"Best overall       : {best_row['Model']} on {best_row['Subset']} "
      f"-- Macro F1 {best_row['Macro F1']:.4f}")
print()

for model_name in sorted(results["Model"].unique()):
    small = pivot_f1.loc[model_name, "Small"]
    large = pivot_f1.loc[model_name, "Large"]
    delta = large - small
    pct_of_data = SUBSET_SIZES["Small"] / SUBSET_SIZES["Large"]
    verdict = (
        "plateaued -- extra data barely helps" if delta < 0.01
        else "still improving -- more data would likely help" if delta > 0.05
        else "modest gains from extra data"
    )
    print(f"{model_name:<22} {small:.4f} -> {large:.4f}  ({delta:+.4f})   {verdict}")
    print(f"{'':22} ({pct_of_data:.0%} of the data recovers "
          f"{small / large:.1%} of the full-data score)")
print("=" * 72)

---

**Outputs written to `Results/`:**

| File | Contents |
|---|---|
| `Tables/data_scaling_results.csv` | One row per (model, subset): all metrics plus timings |
| `Tables/data_scaling_per_class_f1.csv` | Per-class F1 in long form, every model and subset |
| `Tables/data_scaling_gain_summary.csv` | Small → Large improvement per model |
| `Figures/data_scaling_macro_f1.png` | Macro F1 learning curves |
| `Figures/data_scaling_accuracy.png` | Accuracy learning curves |
| `Figures/data_scaling_weighted_f1.png` | Weighted F1 learning curves |
| `Figures/data_scaling_training_time.png` | Training cost vs data size |
| `Figures/data_scaling_rare_class_f1.png` | Rare-class F1 curves |

**Reading the results.** A flat macro-F1 curve means the model saturated early and the remaining
data is redundant — worth saying out loud, because it justifies training on a subset in future work.
A curve that is still climbing at "Large" means the dataset, not the algorithm, is the binding
constraint. The rare-class panel in section 9 usually shows the steepest curves: those classes are
where data volume genuinely matters, and they are what drags macro F1 below accuracy throughout
this project.